In [1]:
%reload_ext autoreload
%autoreload 2

import json
import os
import requests
import numpy as np
import cv2
from PIL import Image
from pymongo import MongoClient
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules
from data_processing.divide_photos import divide_tablet_photo

import signs_alignment as sa
import os

ANNOTATIONS_DIR = os.path.expanduser("~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations")
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser("~/erc-work-data/retrained_models/detr-173/epoch_1000.pth")
SCORE_THRESHOLD = 0.5
Y_THRESHOLD = 35  # for grouping signs into lines
OUTPUT_DIR = "alignment_results"
SAMPLE_LIMIT = 5  # number of samples to process

In [2]:
sa.register_all_modules()
model = init_detector(CONFIG_FILE, CHECKPOINT_FILE, device='cuda:0')

Loads checkpoint by local backend from path: /home/jebediahc/erc-work-data/retrained_models/detr-173/epoch_1000.pth


In [12]:
fragments = sa.get_available_fragments()
print(f"Found {len(fragments)} fragments with both image and annotation")
sample = fragments[0]
print(f"Processing sample: {sample}")

Found 947 fragments with both image and annotation
Processing sample: NBC.4020


In [ ]:
# show groud truth boxes

img = sa.load_image(sample)
gt_boxes = sa.load_ground_truth(sample)

gt_bbox_visualizer = sa.BboxVisualizer(boxes_color=(0, 255, 0)) # green for gt
gt_bbox_visualizer.draw_boxes(img.copy(), gt_boxes)
gt_bbox_visualizer.display_result(vis_opt = "save", path = os.path.join(OUTPUT_DIR, f"debug_{sample}_gt.jpg"))



In [30]:
gt_boxes

[{'bbox': [2336, 1246, 2442, 1408], 'sign_name': 'A'},
 {'bbox': [2463, 1193, 2755, 1398], 'sign_name': 'NA'},
 {'bbox': [2753, 1167, 3065, 1397], 'sign_name': 'BU'},
 {'bbox': [3078, 1164, 3231, 1384], 'sign_name': 'ZA'},
 {'bbox': [3245, 1174, 3492, 1370], 'sign_name': 'ZU'},
 {'bbox': [2281, 1402, 2465, 1591], 'sign_name': 'KI'},
 {'bbox': [3230, 1334, 3506, 1520], 'sign_name': 'UM'},
 {'bbox': [3496, 1305, 3811, 1499], 'sign_name': 'MA'},
 {'bbox': [2782, 1358, 3116, 1557], 'sign_name': 'MA'},
 {'bbox': [2521, 1583, 2843, 1782], 'sign_name': 'GUR'},
 {'bbox': [2296, 1574, 2514, 1787], 'sign_name': 'PI'},
 {'bbox': [2843, 1544, 3209, 1738], 'sign_name': 'TUM'},
 {'bbox': [3196, 1496, 3482, 1711], 'sign_name': 'MA'},
 {'bbox': [3501, 1477, 3731, 1695], 'sign_name': '4'},
 {'bbox': [2337, 1816, 2446, 2034], 'sign_name': 'A'},
 {'bbox': [2436, 1772, 2697, 2016], 'sign_name': 'NA'},
 {'bbox': [2714, 1762, 2881, 1975], 'sign_name': 'A'},
 {'bbox': [2867, 1721, 3075, 1992], 'sign_name': '

In [19]:
# sign text
signs_text = sa.get_signs_from_api(sample)
if signs_text is None:
    raise ValueError(f"No sign text found for sample {sample}")
print(f"  Raw API signs (first 50 chars): {signs_text[:50]}...")

text_lines = sa.parse_api_signs(signs_text)
total_text_signs = sum(len(line) for line in text_lines)
print(f"  Text lines: {len(text_lines)}, total signs: {total_text_signs}")
# print all text lines
text_visualizer = sa.TextVisualizer(text_lines = text_lines)
text_visualizer.write_text_file(filepath = os.path.join(OUTPUT_DIR, f"debug_{sample}_text.txt"), fragment_id=sample)

  Raw API signs (first 50 chars): ABZ579 ABZ70 ABZ371 ABZ586 ABZ6
ABZ461 ABZ214 ABZ3...
  Text lines: 22, total signs: 143


In [32]:
# detect signs
tablet_detector = sa.TabletImageDetector(model, sa.CLASSES, SCORE_THRESHOLD)
detections = tablet_detector.detect(img)

det_bbox_visualizer = sa.BboxVisualizer(boxes_color=(255, 0, 0)) # red for detections
det_bbox_visualizer.draw_boxes(img.copy(), detections)
det_bbox_visualizer.display_result(vis_opt = "save", path = os.path.join(OUTPUT_DIR, f"debug_{sample}_det.jpg"))

In [27]:
detections

[{'bbox': [2405.473876953125,
   571.4232177734375,
   2676.914337158203,
   744.5202331542969],
  'abz_name': 'ABZ13',
  'sign_name': 'AN',
  'score': 0.9370343685150146},
 {'bbox': [3483.4742431640625,
   375.4554748535156,
   3811.85693359375,
   534.3293762207031],
  'abz_name': 'ABZ342',
  'sign_name': 'MA',
  'score': 0.8582826852798462},
 {'bbox': [3263.3095703125,
   521.9877014160156,
   3447.2315673828125,
   709.1443481445312],
  'abz_name': 'ABZ465',
  'sign_name': 'DIN',
  'score': 0.8094568848609924},
 {'bbox': [3040.1115112304688,
   523.5316772460938,
   3258.198486328125,
   708.1777648925781],
  'abz_name': 'ABZ308',
  'sign_name': 'E',
  'score': 0.7991283535957336},
 {'bbox': [2737.27099609375,
   547.1874694824219,
   3021.4111328125,
   726.2962951660156],
  'abz_name': 'ABZ455',
  'sign_name': '|IGI.DIB|',
  'score': 0.7402713894844055},
 {'bbox': [684.7342758178711,
   1991.7841796875,
   882.6572570800781,
   2201.156982421875],
  'abz_name': 'ABZ461',
  'sign_